# ETL Superstore → Star Schema

Three source files are cleaned, combined and loaded into a **star schema** in PostgreSQL.
**Extract, Transform and Load are done with Spark. The database and tables are built with SQL** (files in `workspace/sql/`).

| Step | Tool | Where |
| --- | --- | --- |
| 0. Build database & tables | SQL | pgAdmin: `sql/01_create_database.sql`, `sql/02_create_tables.sql` |
| 1. Extract: read 3 CSV files | Spark | this notebook |
| 2. Transform: clean, join, build dimension & fact | Spark | this notebook |
| 3. Load: insert into the tables | Spark | this notebook |
| 4. Check the result | SQL | pgAdmin: `sql/03_check.sql` |

### From 3 source files to a star schema

```
superstore_customers.csv ──►  dim_customer
superstore_products.csv  ──►  dim_product
superstore_orders.csv    ──►  dim_location, dim_date, fact_sales

                 dim_customer
                      │
   dim_product ── fact_sales ── dim_location
                      │
                   dim_date
```

- **Dimension table** (`dim_`): describes *who / what / where / when*. One row per customer, product, location, date.
- **Fact table** (`fact_`): the events we measure (sales, quantity, profit), with IDs pointing to the dimensions.

## 0. Build the database & tables (SQL, in pgAdmin)

Do this **once, before running this notebook**:

1. pgAdmin → right-click the server → *Query Tool* (connected to database `postgres`) → open `workspace/sql/01_create_database.sql` → run. Database `dwh` is created.
2. Right-click *Databases* → *Refresh* → choose `dwh` → *Query Tool* → open `workspace/sql/02_create_tables.sql` → run. Five empty tables are created, with primary keys and foreign keys.

The column names in `02_create_tables.sql` must match the DataFrames we build below.

### SETUP

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("superstore-etl").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")   # hide Spark warning messages

## 1. Extract

In [ ]:
def read_csv(file_name):
    return spark.read.csv(f'/workspace/data/{file_name}', header=True, inferSchema=True, escape='"')

orders_raw = read_csv('superstore_orders.csv')
customers_raw = read_csv('superstore_customers.csv')
products_raw = read_csv('superstore_products.csv')

print('orders:', orders_raw.count(), '| customers:', customers_raw.count(), '| products:', products_raw.count())

## 2. Transform

### Step 1. Rename columns to snake_case

`Customer ID` → `customer_id`. Column names without spaces are easier to use in Spark, and they must match the SQL tables.
pandas: `df.columns = [...]` → Spark: `df.toDF(*new_names)`

In [ ]:
def to_snake_case(df):
    new_names = [c.lower().replace(' ', '_').replace('-', '_').replace('/', '_') for c in df.columns]
    return df.toDF(*new_names)

orders = to_snake_case(orders_raw)
customers = to_snake_case(customers_raw)
products = to_snake_case(products_raw)

print(orders.columns)
print(customers.columns)
print(products.columns)

### Step 2. Check data quality

**Duplicate rows** → pandas: `df.duplicated().sum()`

In [ ]:
for name, df in [('orders', orders), ('customers', customers), ('products', products)]:
    print(name, df.count() - df.dropDuplicates().count())

**Missing values in orders** → pandas: `df.isna().sum()`

In [ ]:
for c in orders.columns:
    n = orders.filter(F.col(c).isNull()).count()
    if n > 0:
        print(c, n)

**Inconsistent text**

In [ ]:
customers.groupBy('segment').count().show()
products.groupBy('category').count().show()
orders.select('region').distinct().show()

**Product IDs with more than one name** in the products file:

In [ ]:
products.groupBy('product_id').agg(F.countDistinct('product_name').alias('names')).filter('names > 1').show(3)

**Postal codes that lost their leading zero** (`6824` should be `06824`) and **dates still as text**:

In [ ]:
orders.filter(F.length(F.col('postal_code').cast('string')) < 5).select('city', 'state', 'postal_code').show(3)
orders.select('order_date', 'ship_date').printSchema()

### Step 3. Clean each table

| Problem | Fix | pandas |
| --- | --- | --- |
| duplicate rows | `dropDuplicates()` | `drop_duplicates()` |
| extra spaces & mixed case | `F.initcap(F.trim(col))` | `.str.strip().str.title()` |
| one ID, several names | `dropDuplicates(['product_id'])` keeps one row per ID | `drop_duplicates('Product ID')` |
| postal code without leading zero | `F.lpad(col, 5, '0')` | `.str.zfill(5)` |
| missing postal code | `fillna({'postal_code': 'Unknown'})` | `fillna(...)` |
| dates as text | `F.to_date(col, 'M/d/yyyy')` | `pd.to_datetime(col, format='%m/%d/%Y')` |
| missing sales | `dropna(subset=['sales'])` | `dropna(subset=[...])` |

In [ ]:
customers_clean = (customers
    .dropDuplicates()
    .withColumn('customer_name', F.trim('customer_name'))
    .withColumn('segment', F.initcap(F.trim('segment')))
    .dropDuplicates(['customer_id']))

products_clean = (products
    .dropDuplicates()
    .withColumn('category', F.initcap(F.trim('category')))
    .dropDuplicates(['product_id']))

orders_clean = (orders
    .dropDuplicates()
    .withColumn('region', F.initcap(F.trim('region')))
    .withColumn('postal_code', F.lpad(F.col('postal_code').cast('string'), 5, '0'))
    .fillna({'postal_code': 'Unknown'})
    .withColumn('order_date', F.to_date('order_date', 'M/d/yyyy'))
    .withColumn('ship_date', F.to_date('ship_date', 'M/d/yyyy'))
    .dropna(subset=['sales']))

print('customers:', customers.count(), '->', customers_clean.count())
print('products :', products.count(), '->', products_clean.count())
print('orders   :', orders.count(), '->', orders_clean.count())
customers_clean.groupBy('segment').count().show()

### Step 4. Join the sources

Before building the fact table, check that every order points to a customer and a product that exist.
`left_anti` returns the orders **without** a match.

In [ ]:
missing_customer = orders_clean.join(customers_clean, on='customer_id', how='left_anti')
missing_product = orders_clean.join(products_clean, on='product_id', how='left_anti')

print('orders without customer:', missing_customer.count())
print('orders without product :', missing_product.count())
missing_product.select('row_id', 'order_id', 'product_id', 'sales').show()

Two orders refer to products that are not in the products file. The foreign key in `fact_sales` would reject them,
so we keep only orders with a known product using an **inner join** (only matching rows are kept).
In a real project, report these rows to the data owner.

In [ ]:
orders_valid = orders_clean.join(products_clean.select('product_id'), on='product_id', how='inner')
print('orders:', orders_clean.count(), '->', orders_valid.count())

**Combine all sources** into one table, like the original export, to see the data together.
The row count must stay the same as `orders_valid`; if it grows, a source still has duplicate IDs.

In [ ]:
sales_full = (orders_valid
    .join(customers_clean, on='customer_id', how='left')
    .join(products_clean, on='product_id', how='left'))

print('rows:', sales_full.count())
sales_full.groupBy('segment', 'category').agg(F.round(F.sum('sales'), 2).alias('total_sales')).orderBy('segment', 'category').show()

### Step 5. Build the dimension tables

`dim_customer` and `dim_product` come straight from the cleaned source files: one row per ID.

In [ ]:
dim_customer = customers_clean.select('customer_id', 'customer_name', 'segment')
dim_product = products_clean.select('product_id', 'product_name', 'category', 'sub_category')

dim_customer.show(3)
dim_product.show(3, truncate=False)

A location has no ID in the source files, so we create one: take the unique locations, sort them, and number them 1, 2, 3, ...

In [ ]:
location_cols = ['country_region', 'region', 'state', 'city', 'postal_code']

dim_location = (orders_valid.select(location_cols).dropDuplicates()
                .withColumn('location_id', F.row_number().over(Window.orderBy('state', 'city', 'postal_code'))))
dim_location.show(3)

The date dimension has one row per date, with parts that are useful for reports.
`date_id` is the date as a number, for example `20201108`.

In [ ]:
dates = (orders_valid.select(F.col('order_date').alias('full_date'))
         .union(orders_valid.select(F.col('ship_date').alias('full_date')))
         .distinct())

dim_date = (dates
    .withColumn('date_id', F.date_format('full_date', 'yyyyMMdd').cast('int'))
    .withColumn('year', F.year('full_date'))
    .withColumn('quarter', F.quarter('full_date'))
    .withColumn('month', F.month('full_date'))
    .withColumn('month_name', F.date_format('full_date', 'MMMM'))
    .withColumn('day_name', F.date_format('full_date', 'EEEE')))
dim_date.orderBy('full_date').show(3)

### Step 6. Build the fact table

The fact table keeps the numbers (`sales`, `quantity`, `discount`, `profit`) and only the **IDs** of each dimension.
`location_id` is taken by joining back to `dim_location`.

In [ ]:
fact_sales = (orders_valid
    .join(dim_location, on=location_cols, how='left')
    .withColumn('order_date_id', F.date_format('order_date', 'yyyyMMdd').cast('int'))
    .withColumn('ship_date_id', F.date_format('ship_date', 'yyyyMMdd').cast('int'))
    .select('row_id', 'order_id', 'order_date_id', 'ship_date_id', 'customer_id', 'product_id',
            'location_id', 'ship_mode', 'sales', 'quantity', 'discount', 'profit'))
fact_sales.show(3)

**Check:** one fact row per valid order, and no missing IDs.

In [ ]:
print('valid orders:', orders_valid.count(), '| fact rows:', fact_sales.count())
print('missing location_id:', fact_sales.filter(F.col('location_id').isNull()).count())

for name, table in [('dim_customer', dim_customer), ('dim_product', dim_product),
                    ('dim_location', dim_location), ('dim_date', dim_date), ('fact_sales', fact_sales)]:
    print(name, table.count())

## 3. Load to PostgreSQL

The DataFrames are inserted into the tables we created with SQL in step 0, in **PostgreSQL on your laptop**.

| | Host | Port |
| --- | --- | --- |
| pgAdmin (on the laptop) | `localhost` | `5432` |
| This notebook (inside the container) | `host.docker.internal` | `5432` |

Inside the container, `localhost` means the container itself, so we use `host.docker.internal` to reach the laptop.
The host, user and password come from the `.env` file, so the password is **never written in the notebook**.

In [ ]:
missing = [k for k in ('PG_HOST', 'PG_PORT', 'PG_DB', 'PG_USER', 'PG_PASSWORD') if not os.getenv(k)]
if missing:
    raise SystemExit(f'Not set in .env: {missing}. Copy .env.example to .env, fill it in, then run: docker compose up -d')

jdbc_url = f"jdbc:postgresql://{os.environ['PG_HOST']}:{os.environ['PG_PORT']}/{os.environ['PG_DB']}"
jdbc_props = {'user': os.environ['PG_USER'], 'password': os.environ['PG_PASSWORD'], 'driver': 'org.postgresql.Driver'}
print(jdbc_url)

**Check before loading**: the connection works, the tables from `02_create_tables.sql` exist, and they are still empty
(loading twice would create duplicate keys).

In [ ]:
import psycopg2

try:
    conn = psycopg2.connect(host=os.environ['PG_HOST'], port=os.environ['PG_PORT'], dbname=os.environ['PG_DB'],
                            user=os.environ['PG_USER'], password=os.environ['PG_PASSWORD'], connect_timeout=5)
except psycopg2.OperationalError as e:
    msg = str(e)
    if 'does not exist' in msg:
        raise SystemExit(f"Database {os.environ['PG_DB']} does not exist. Run sql/01_create_database.sql in pgAdmin first.")
    if 'password authentication failed' in msg:
        raise SystemExit('Wrong password. PG_USER / PG_PASSWORD in .env must match pgAdmin.')
    if 'Connection refused' in msg or 'timeout' in msg:
        raise SystemExit('Cannot reach PostgreSQL on the laptop. Make sure it is running (pgAdmin can connect).')
    raise

with conn.cursor() as cur:
    cur.execute("SELECT to_regclass('fact_sales')")
    if cur.fetchone()[0] is None:
        raise SystemExit('Table fact_sales does not exist. Run sql/02_create_tables.sql in pgAdmin first.')
    cur.execute('SELECT COUNT(*) FROM fact_sales')
    if cur.fetchone()[0] > 0:
        raise SystemExit('Tables already contain data. Run sql/02_create_tables.sql again to empty them, then run this cell again.')
conn.close()
print('Connected. Tables exist and are empty.')

**Insert the data.** Dimensions first, then the fact table, because the fact table has foreign keys to the dimensions.

`mode='append'` inserts rows into the existing tables and keeps the design from `02_create_tables.sql`.
(`mode='overwrite'` would drop our tables and let Spark create new ones without primary & foreign keys.)

In [ ]:
tables = [('dim_customer', dim_customer), ('dim_product', dim_product), ('dim_location', dim_location),
          ('dim_date', dim_date), ('fact_sales', fact_sales)]

for name, table in tables:
    table.write.jdbc(jdbc_url, name, mode='append', properties=jdbc_props)
    print('loaded', name)

## 4. Check the result

Read back from PostgreSQL with a query that joins the fact table to two dimensions.
More queries are in `sql/03_check.sql` (run it in pgAdmin).

In [ ]:
query = '''
(SELECT l.region, p.category, ROUND(SUM(f.sales), 2)::float AS total_sales
 FROM fact_sales f
 JOIN dim_location l ON f.location_id = l.location_id
 JOIN dim_product  p ON f.product_id  = p.product_id
 GROUP BY l.region, p.category
 ORDER BY l.region, p.category) AS t
'''
spark.read.jdbc(jdbc_url, query, properties=jdbc_props).show()

Done. Stop Spark to free the laptop's memory.

In [ ]:
spark.stop()